In [ ]:
%pip install selenium
%pip install webdriver-manager
%pip install tqdm

## 모듈화한 함수 불러서 쓰기(각 링크 txt저장까지)

In [2]:
import sys
from pathlib import Path

# 현재 노트북 위치 (AC 폴더)
CURRENT_DIR = Path.cwd()

# 무조건 한 칸 위 (data_crawling)
BASE_DIR = CURRENT_DIR.parent

# import 경로 추가 (맨 앞에 넣는 게 더 안전)
sys.path.insert(0, str(BASE_DIR))

from selenium_auto_module import full_page, product_links


URL = "https://www.lge.co.kr/category/vacuum-cleaners"


html_sample = full_page(URL, "vacuum")
print(html_sample)

links_sample = product_links("vacuum")
print(links_sample)

페이지 로딩 완료
1번째 클릭
끝까지 펼침 완료
HTML 길이: 637388
HTML 저장 완료: vacuum_full_page_html.txt
<html lang="ko"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1"><link rel="preload" as="image" href="/kr/common/footer/sns_youtube.svg"><link rel="preload" as="image" href="/kr/common/footer/sns_insta.svg"><link rel="preload" as="image" href="/kr/common/footer/sns_facebook.svg"><link rel="preload" as="image" href="/kr/common/footer/sns_newsroom.svg"><link rel="preload" as="image" href="/kr/common/footer/sns_kakao.svg"><link rel="preload" as="image"
제품 카드 개수: 34
추출된 링크 개수: 34
저장 완료: vacuum_product_links.txt
['https://www.lge.co.kr/vacuum-cleaners/b94ahb-1', 'https://www.lge.co.kr/vacuum-cleaners/c33fnt', 'https://www.lge.co.kr/vacuum-cleaners/mo972wa', 'https://www.lge.co.kr/vacuum-cleaners/ax920bwe-bkor1', 'https://www.lge.co.kr/vacuum-cleaners/a737wa']


## 재원추출 및 csv저장

In [ ]:
import time
import re
import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager


# =========================
# 테스트 URL 1개
# =========================
URL = "https://www.lge.co.kr/vacuum-cleaners/ax948bhe-bkor1"


driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install())
)

wait = WebDriverWait(driver, 10)


def clean_text(text):
    if not text:
        return ""
    return re.sub(r"\s+", " ", text).strip()


def get_product_name(soup):
    tag = soup.select_one("h1.name")
    if not tag:
        return ""

    for child in tag.select("span, em"):
        child.decompose()

    return clean_text(tag.get_text())


def get_image_url(soup):
    img = soup.select_one("#base_detail_target")
    if not img:
        return ""

    src = img.get("src", "").strip()
    if not src:
        return ""

    if src.startswith("http"):
        return src

    return "https://www.lge.co.kr" + src


def get_price(soup):
    tag = soup.select_one("small.original")
    if not tag:
        return ""

    for blind in tag.select(".blind"):
        blind.decompose()

    return clean_text(tag.get_text())


def get_spec(soup, *labels):
    spec_area = soup.select_one("#pdp_spec") or soup

    for dt in spec_area.find_all("dt"):
        dt_text = clean_text(dt.get_text(" ", strip=True))

        for label in labels:
            if label in dt_text:
                dd = dt.find_next_sibling("dd")
                if dd:
                    return clean_text(dd.get_text(" ", strip=True))

    return ""


def get_spec_in_box(soup, box_keyword, *labels):
    spec_area = soup.select_one("#pdp_spec") or soup

    for box in spec_area.select("div.box"):
        h3 = box.select_one("h3.tit")
        box_title = clean_text(h3.get_text(" ", strip=True)) if h3 else ""

        if box_keyword in box_title:
            for dt in box.find_all("dt"):
                dt_text = clean_text(dt.get_text(" ", strip=True))

                for label in labels:
                    if label in dt_text:
                        dd = dt.find_next_sibling("dd")
                        if dd:
                            return clean_text(dd.get_text(" ", strip=True))

    return ""


def get_body_size(soup):
    value = get_spec_in_box(
        soup,
        "청소기 본체",
        "크기",
        "크기 (WxHxD",
        "크기(WxHxD"
    )

    if value:
        return value

    return get_spec_in_box(
        soup,
        "크기 & 무게",
        "크기",
        "크기 (WxHxD",
        "크기(WxHxD"
    )


def get_body_weight(soup):
    value = get_spec_in_box(
        soup,
        "청소기 본체",
        "무게"
    )

    if value:
        return value

    return get_spec_in_box(
        soup,
        "크기 & 무게",
        "무게"
    )


def get_tower_size(soup):
    return get_spec_in_box(
        soup,
        "올인원타워",
        "크기",
        "크기 (WxHxD",
        "크기(WxHxD"
    )


def get_tower_weight(soup):
    return get_spec_in_box(
        soup,
        "올인원타워",
        "무게"
    )


driver.get(URL)
time.sleep(3)

try:
    more_btn = wait.until(
        EC.element_to_be_clickable(
            (
                By.XPATH,
                "//a[contains(@title,'펼치기') or contains(., '제품 정보 더 보기') or contains(., '제품스펙 상세정보 펼치기')]"
            )
        )
    )

    driver.execute_script(
        "arguments[0].scrollIntoView({block:'center'});",
        more_btn
    )
    time.sleep(1)

    driver.execute_script("arguments[0].click();", more_btn)
    time.sleep(2)

except:
    pass


soup = BeautifulSoup(driver.page_source, "html.parser")


row = {
    "제품명": get_product_name(soup),
    "이미지 링크": get_image_url(soup),
    "가격(원)": get_price(soup),

    "에너지 소비효율 등급": get_spec(
        soup,
        "에너지소비효율등급",
        "에너지 소비효율등급",
        "에너지소비효율 등급",
        "에너지 소비효율 등급"
    ),

    "정격소비전력(W)": get_spec(
        soup,
        "정격소비전력",
        "정격소비전력 (W)",
        "소비전력",
    ),

    "크기(청소기 본체)": get_body_size(soup),
    "크기(올인원 타워)": get_tower_size(soup),

    "무게(청소기 본체)": get_body_weight(soup),
    "무게(올인원 타워)": get_tower_weight(soup),

    "색상(청소기 본체)": get_spec(
        soup,
        "색상 (청소기 본체)",
        "색상(청소기 본체)",
        "색상"
    ),

    "최대 흡입력(W)": get_spec(
        soup,
        "최대 흡입력",
        "최대흡입력",
        "흡입력"
    ),

    "배터리 포함 수(개)": get_spec(
        soup,
        "배터리 포함 수",
        "배터리포함수",
        "배터리 수",
        "배터리"
    ),
}


df = pd.DataFrame([row])
df.to_csv("lge_vacuum_one_product.csv", index=False, encoding="utf-8-sig")

print("저장 완료: lge_vacuum_one_product.csv")
display(df)

driver.quit()

저장 완료: lge_vacuum_one_product.csv


,제품명,이미지 링크,가격(원),에너지 소비효율 등급,소비전력(W)(청소시),크기(청소기 본체),크기(올인원 타워),무게(청소기 본체),무게(올인원 타워),색상(청소기 본체),최대 흡입력(W),배터리 포함 수(개)
0,LG 코드제로 오브제컬렉션 A9,https://www.lge.co.kr/kr/images/vacuum-cleaner...,"1,580,000원",해당없음,400,"300 x 1,100 x 245","256 x 1,009 x 298",2.49,9.2,에센스 화이트,280,2 (듀얼)


In [4]:
import time
import re
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager


driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install())
)

wait = WebDriverWait(driver, 10)


# =========================
# 청소기 상품 링크 txt 읽기
# =========================
with open("vacuum_product_links.txt", "r", encoding="utf-8") as f:
    product_links = [line.strip() for line in f if line.strip()]

print("총 링크 개수:", len(product_links))


# =========================
# 공통 함수
# =========================
def clean_text(text):
    if not text:
        return ""
    return re.sub(r"\s+", " ", text).strip()


def get_product_name(soup):
    tag = soup.select_one("h1.name")
    if not tag:
        return ""

    for child in tag.select("span, em"):
        child.decompose()

    return clean_text(tag.get_text())


def get_image_url(soup):
    img = soup.select_one("#base_detail_target")
    if not img:
        return ""

    src = img.get("src", "").strip()
    if not src:
        return ""

    if src.startswith("http"):
        return src

    return "https://www.lge.co.kr" + src


def get_price(soup):
    tag = soup.select_one("small.original")
    if not tag:
        return ""

    for blind in tag.select(".blind"):
        blind.decompose()

    return clean_text(tag.get_text())


def get_spec(soup, *labels):
    spec_area = soup.select_one("#pdp_spec") or soup

    for dt in spec_area.find_all("dt"):
        dt_text = clean_text(dt.get_text(" ", strip=True))

        for label in labels:
            if label in dt_text:
                dd = dt.find_next_sibling("dd")
                if dd:
                    return clean_text(dd.get_text(" ", strip=True))

    return ""


# =========================
# 특정 box 안에서만 dt/dd 추출
# 예: 크기 & 무게 (청소기 본체)
# =========================
def get_spec_in_box(soup, box_keyword, *labels):
    spec_area = soup.select_one("#pdp_spec") or soup

    for box in spec_area.select("div.box"):
        h3 = box.select_one("h3.tit")
        box_title = clean_text(h3.get_text(" ", strip=True)) if h3 else ""

        if box_keyword in box_title:
            for dt in box.find_all("dt"):
                dt_text = clean_text(dt.get_text(" ", strip=True))

                for label in labels:
                    if label in dt_text:
                        dd = dt.find_next_sibling("dd")
                        if dd:
                            return clean_text(dd.get_text(" ", strip=True))

    return ""


# =========================
# 청소기 본체 크기/무게
# - "청소기 본체" box가 있으면 거기서 추출
# - 없으면 그냥 "크기 & 무게" box를 청소기 본체로 간주
# =========================
def get_body_size(soup):
    value = get_spec_in_box(
        soup,
        "청소기 본체",
        "크기",
        "크기 (WxHxD",
        "크기(WxHxD"
    )

    if value:
        return value

    return get_spec_in_box(
        soup,
        "크기 & 무게",
        "크기",
        "크기 (WxHxD",
        "크기(WxHxD"
    )


def get_body_weight(soup):
    value = get_spec_in_box(
        soup,
        "청소기 본체",
        "무게"
    )

    if value:
        return value

    return get_spec_in_box(
        soup,
        "크기 & 무게",
        "무게"
    )


def get_tower_size(soup):
    return get_spec_in_box(
        soup,
        "올인원타워",
        "크기",
        "크기 (WxHxD",
        "크기(WxHxD"
    )


def get_tower_weight(soup):
    return get_spec_in_box(
        soup,
        "올인원타워",
        "무게"
    )

def get_manual_pdf_url(soup):
    for top_area in soup.select("div.top-area"):
        strong = top_area.select_one("strong")
        title = clean_text(strong.get_text()) if strong else ""

        # "제품 사용 설명서(바로보기용)" 제외
        if title == "제품 사용 설명서":
            parent = top_area.find_parent()

            if parent:
                a_tag = parent.select_one("div.btn-group a[href]")

                if a_tag:
                    href = a_tag.get("href", "").strip()

                    if href.startswith("http"):
                        return href
                    elif href.startswith("/"):
                        return "https://www.lge.co.kr" + href
                    else:
                        return href

    return ""

# =========================
# 전체 링크 순회
# =========================
rows = []
failures = []

pbar = tqdm(product_links, desc="청소기 상품 크롤링", ncols=120)

for idx, url in enumerate(pbar, start=1):
    try:
        pbar.set_postfix_str(f"{idx}/{len(product_links)} 접속 중")

        driver.get(url)
        time.sleep(3)

        # 제품 정보 / 제품 스펙 더 보기 클릭
        try:
            more_btn = wait.until(
                EC.element_to_be_clickable(
                    (
                        By.XPATH,
                        "//a[contains(@title,'펼치기') or contains(., '제품 정보 더 보기') or contains(., '제품스펙 상세정보 펼치기')]"
                    )
                )
            )

            driver.execute_script(
                "arguments[0].scrollIntoView({block:'center'});",
                more_btn
            )
            time.sleep(1)

            driver.execute_script("arguments[0].click();", more_btn)
            time.sleep(2)

        except:
            pass

        soup = BeautifulSoup(driver.page_source, "html.parser")

        row = {
            "제품명": get_product_name(soup),
            "이미지 링크": get_image_url(soup),
            "가격(원)": get_price(soup),

            "에너지 소비효율 등급": get_spec(
                soup,
                "에너지소비효율등급",
                "에너지 소비효율등급",
                "에너지소비효율 등급",
                "에너지 소비효율 등급"
            ),

                    
            "정격소비전력(W)": get_spec(
                soup,
                "정격소비전력",
                "정격소비전력 (W)",
                "소비전력",
            ),

            "크기(청소기 본체)": get_body_size(soup),
            "크기(올인원 타워)": get_tower_size(soup),

            "무게(청소기 본체)": get_body_weight(soup),
            "무게(올인원 타워)": get_tower_weight(soup),

            "색상(청소기 본체)": get_spec(
                soup,
                "색상 (청소기 본체)",
                "색상(청소기 본체)",
                "색상"
            ),

            "최대 흡입력(W)": get_spec(
                soup,
                "최대 흡입력",
                "최대흡입력",
                "흡입력"
            ),

            "배터리 포함 수(개)": get_spec(
                soup,
                "배터리 포함 수",
                "배터리포함수",
                "배터리 수",
                "배터리"
            ),
            
            "제품 사용 설명서": get_manual_pdf_url(soup),
        }

        rows.append(row)

        pbar.set_postfix_str(f"완료: {row['제품명'][:25]}")

        if idx % 10 == 0:
            pd.DataFrame(rows).to_csv(
                "lge_vacuum_temp.csv",
                index=False,
                encoding="utf-8-sig"
            )

    except Exception as e:
        failures.append({"url": url, "error": str(e)})
        pbar.set_postfix_str(f"실패: {idx}/{len(product_links)}")


# =========================
# 최종 CSV 저장
# =========================
df = pd.DataFrame(rows)

df.to_csv(
    "lge_vacuum_all_products.csv",
    index=False,
    encoding="utf-8-sig"
)

print("전체 저장 완료: lge_vacuum_all_products.csv")
print("총 수집 개수:", len(df))
print("실패 개수:", len(failures))

display(df.head())


if failures:
    pd.DataFrame(failures).to_csv(
        "lge_vacuum_failures.csv",
        index=False,
        encoding="utf-8-sig"
    )
    print("실패 목록 저장 완료: lge_vacuum_failures.csv")


driver.quit()

총 링크 개수: 34


청소기 상품 크롤링: 100%|███████████████████████████████████████| 34/34 [07:49<00:00, 13.82s/it, 완료: LG 업소용 청소기]접속 중]]s]

전체 저장 완료: lge_vacuum_all_products.csv
총 수집 개수: 34
실패 개수: 0


,제품명,이미지 링크,가격(원),에너지 소비효율 등급,정격소비전력(W),크기(청소기 본체),크기(올인원 타워),무게(청소기 본체),무게(올인원 타워),색상(청소기 본체),최대 흡입력(W),배터리 포함 수(개),제품 사용 설명서
0,LG 코드제로 오브제컬렉션 A7 Core,https://www.lge.co.kr/kr/images/vacuum-cleaner...,"900,000원",해당없음,350,"250 x 1,120 x 260","255 x 1,009 x 297",2.47,9.7,카밍 베이지,220,1 (싱글),https://gscs-b2c.lge.com/open/downloadFile?fil...
1,LG 코드제로 오브제컬렉션 A7 Core,https://www.lge.co.kr/kr/images/vacuum-cleaner...,,해당없음,350,"250 x 1,120 x 260","255 x 1,009 x 297",2.47,9.7,카밍 베이지,220,1 (싱글),https://gscs-b2c.lge.com/open/downloadFile?fil...
2,LG 코드제로 오브제컬렉션 A7 Core,https://www.lge.co.kr/kr/images/vacuum-cleaner...,"1,041,000원",해당없음,350,"250 x 1,120 x 260","255 x 1,009 x 297",2.47,9.7,카밍 베이지,220,1 (싱글),https://gscs-b2c.lge.com/open/downloadFile?fil...
3,LG 코드제로 오브제컬렉션 A7 Core,https://www.lge.co.kr/kr/images/vacuum-cleaner...,"1,130,000원",해당없음,350,"250 x 1,120 x 260","255 x 1,009 x 297",2.47,9.7,카밍 베이지,220,1 (싱글),https://gscs-b2c.lge.com/open/downloadFile?fil...
4,LG 코드제로 AI 오브제컬렉션 A9,https://www.lge.co.kr/kr/images/vacuum-cleaner...,"1,331,000원",해당없음,400,"300 x 1,100 x 245","256 x 1,009 x 298",2.54,9.2,크림 스카이,320,1 (싱글),https://gscs-b2c.lge.com/open/downloadFile?fil...
